<a href="https://colab.research.google.com/github/seo104/26-2-ESAA/blob/main/ESAA_12%EC%A3%BC%EC%B0%A8_%EC%88%98%EC%83%81%EC%9E%91_%EB%A6%AC%EB%B7%B0_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#수상작 리뷰
---
##[식음업장 메뉴 수요 예측 AI 온라인 해커톤]
https://dacon.io/competitions/official/236559/overview/description

---
###데이터
train :
2023.01.01 ~ 2024.06.15의 영업장명_메뉴명별 매출 수량 정보
- 영업일자
- 영업장명_메뉴명
- 매출수량

test :
2025년의 특정 시점(28일)의 영업장명_메뉴명별 매출 수량 정보
- 영업일자
- 영업장명_메뉴명
- 매출수량

sample_submission.csv - 제출 양식

---
###코드 흐름
1. 데이터 전처리 및 메뉴 분류
- 메뉴명과 영업장명을 활용하기 위해 도메인 지식을 기반으로 menu.csv 생성
- 메뉴 종류 및 영업장 유형을 구분하여 모델 학습에 활용
2. 피처 엔지니어링
- 과거 28일의 판매량을 활용하여 학습 데이터 구성
- 이동평균, 판매량 표준편차, 추세 등 시계열 파생변수 생성
- 요일, 월, 계절 등 날짜 관련 피처 추가
- 메뉴 및 영업장 특성을 반영한 도메인 기반 피처 생성
- 이상치로 인한 영향을 줄이기 위해 Winsorization 적용
3. 1~4차 메타 모델 학습
- GRU, Transformer, LSTM, TCN 등 딥러닝 모델과 LightGBM, XGBoost 등 트리 기반 모델 활용
- 4가지 메타 모델을 각각 구성하여 다양한 모델 조합으로 예측 수행
- 5-Fold 교차검증을 통해 모델별 성능 평가 및 최적 가중치 탐색
4. 1차 앙상블
- 각 메타 모델 내부에서 여러 모델의 예측값을 결합
- 모델별 예측 가중치를 적용하여 최종 예측값 생성
- 예측값의 급격한 변화를 줄이기 위해 Smoothing 및 후처리 적용
5. 2차 앙상블 및 제출 파일 생성
- 4개의 메타 모델이 생성한 submission 파일을 불러옴
- 각 모델의 예측값을 동일한 가중치로 평균하여 최종 예측값 생성
- 최종 submission.csv 파일 저장
---
###주요 코드 분석
① 과거 판매량을 활용한 시계열 파생변수 생성

In [ ]:
mean7  = hist[-7:].mean()
mean14 = hist[-14:].mean()
mean28 = hist.mean()

std7  = hist[-7:].std()
std14 = hist[-14:].std()
std28 = hist.std()

lag1 = hist[-1]
lag7 = hist[-7]
trend = linear_trend(hist)

- 최근 7일, 14일, 28일의 평균과 표준편차 계산
- lag를 활용하여 1일 전, 7일 전 판매량 정보 추가
- 선형 추세를 통해 최근 판매량의 증가 및 감소 경향 반영
- 단순 판매량뿐 아니라 판매 패턴과 변동성까지 모델이 학습하도록 피처 구성

② 날짜의 주기성을 반영한 캘린더 피처 생성

In [ ]:
sin_dow = np.sin(2*np.pi*dow/7)
cos_dow = np.cos(2*np.pi*dow/7)

sin_doy = np.sin(2*np.pi*doy/365.25)
cos_doy = np.cos(2*np.pi*doy/365.25)

- 요일과 연중 날짜를 sin, cos 함수로 변환
- 요일 및 계절에 따른 반복적인 판매 패턴을 연속적인 값으로 표현
- 날짜의 주기성을 모델이 효과적으로 학습할 수 있도록 피처 생성

③ 2단계 앙상블 적용


In [ ]:
for path in cand:
    df = pd.read_csv(path)
    aligned = sample[[key_col]].merge(df, on=key_col, how="left")
    vals = aligned[item_cols].apply(
        pd.to_numeric, errors="coerce"
    ).to_numpy()

    sum_mat += np.nan_to_num(vals)
    cnt_mat += np.isfinite(vals)

avg = sum_mat / np.clip(cnt_mat, 1.0, None)

- 4개의 메타 모델이 생성한 제출 파일을 불러와 날짜와 메뉴 기준으로 정렬
- 각 모델의 예측값을 동일한 가중치로 평균 계산
- 결측값은 평균 계산에서 제외하여 최종 예측값 생성
- 개별 모델의 예측 결과를 한 번 더 결합하는 2단계 앙상블 적용
---
###새롭게 알게 된 내용 / 배울 점
1. 최근 7일, 14일, 28일의 평균과 표준편차, 과거 판매량 등을 활용하여 시계열 데이터의 추세와 변동성을 피처로 만드는 방법을 알게 됨.
2. 개별 모델 내부의 앙상블뿐 아니라 여러 모델의 제출 결과를 다시 결합하는 2단계 앙상블을 통해 예측 성능을 개선하는 방법을 알게 됨.